In [3]:
import cv2
import torch
import numpy as np
from PIL import Image
from models.network import SCI  # THIS IS CRITICAL: Use the correct import!

# Initialize the SCI model and load pretrained weights
model = SCI().cpu()  # Use .cpu() if you don't have CUDA
model.load_state_dict(torch.load("weights\easy.pt"))
model.eval()

# Set up camera
cap = cv2.VideoCapture(0)  # Adjust if your camera is not device 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert BGR to RGB, scale to [0,1], convert to tensor, and add batch and channel dimensions
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = (frame_rgb / 255.0).astype(np.float32)
    input_tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).cuda()

    # Enhance with SCI
    with torch.no_grad():
        enhanced_tensor = model(input_tensor)

    # Convert back to numpy, scale, and ensure correct type and color space for OpenCV
    enhanced_img = (enhanced_tensor.squeeze().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    enhanced_bgr = cv2.cvtColor(enhanced_img, cv2.COLOR_RGB2BGR)

    # Display original and enhanced frames
    cv2.imshow('Original', frame)
    cv2.imshow('Enhanced (SCI)', enhanced_bgr)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


ModuleNotFoundError: No module named 'models'